In [33]:
import json
from elasticsearch import Elasticsearch
from elasticsearch.helpers import bulk

In [4]:
%pip install elasticsearch

Note: you may need to restart the kernel to use updated packages.


In [37]:
# client = Elasticsearch("http://localhost:9200")


# import warnings
# from elasticsearch.exceptions import ElasticsearchWarning

# # To hide messages for HTTPS
# warnings.simplefilter('ignore', ElasticsearchWarning)
# warnings.filterwarnings("ignore", message="Unverified HTTPS request")

# # Connection to Elasticsearch 8.x
# client = Elasticsearch(
#     "https://localhost:9200",
#     basic_auth=("elastic", "lcruhVaFiAjkVXltXOV2"),
#     verify_certs=False
# )

# # # Confirm connection
# print(client.info())

C:\Users\monoh\anaconda3\Lib\site-packages\elasticsearch\_sync\client\__init__.py:329: SecurityWarning: Connecting to 'https://localhost:9200' using TLS with verify_certs=False is insecure
  _transport = transport_class(


{'name': 'LAPTOP-R9IG9QUF', 'cluster_name': 'elasticsearch', 'cluster_uuid': 'zdYT7L05SQyrj-ptW1rzvA', 'version': {'number': '9.5.6', 'build_flavor': 'default', 'build_type': 'zip', 'build_hash': 'fd1e8011342a0988e432b999cea4bf884ca120ed', 'build_date': '2026-10-28T22:37:54.387301573Z', 'build_snapshot': False, 'lucene_version': '10.5.2', 'minimum_wire_compatibility_version': '8.19.0', 'minimum_index_compatibility_version': '8.0.0'}, 'tagline': 'You Know, for Search'}


In [39]:
all_documents = []

file_path = "IR2026/documents.jsonl"

# Open jsonl file for reading
with open(file_path, "r", encoding="utf-8") as file:
    for line in file:
        # Convert each line from JSON string to a Python dictionary
        json_data = json.loads(line)
        all_documents.append(json_data)

print(f"{len(all_documents)} Successfully read!")

if len(all_documents) > 0:
    print("\nFirst document's structure:")
    print(all_documents[0])

40221 Successfully read!

First document's structure:
{'_id': '11547', 'title': 'Is there a word meaning "an unwanted eponym"?', 'text': 'An eponym is one way to eternal (if posthumous) fame. But is there a word meaning an eponym someone would sooner not have? (One would presume that Captain Charles _Boycott_ , Mr Justice _Lynch_ , and Patrick _Hooligan_ would not appreciate their undying notoriety.)'}


In [41]:
index_name = "ir2026_index"

# Delete the index if it already exists
if client.indices.exists(index=index_name):
    client.indices.delete(index=index_name)

# Define Stucture (Mapping & Settings)
mapping = {
    "settings": {
        "number_of_shards": 1,
        "analysis": {
            "analyzer": {
                "default": {
                    "type": "english" # Defines stemming and stopwords removal
                }
            }
        }
    },
    "mappings": {
        "properties": {
            "title": {"type": "text"},
            "text": {"type": "text"}
        }
    }
}

# Create Index 
client.indices.create(index=index_name, body=mapping)
print(f"The index '{index_name}' was created.")

# Prepare data for Bulk Insert
bulk_documents = []
for doc in all_documents:
    bulk_documents.append({
        "_index": index_name,
        "_id": doc["_id"],  #Keep the initial ID to match with the qrels
        "_source": {
            "title": doc["title"],
            "text": doc["text"]
        }
    })


# Bulk Insert Of the documents to ElasticSearch
print("Data Insert is starting. Please wait...")
bulk(client, bulk_documents, refresh=True)

# Confirmation
count = client.cat.count(index=index_name, format="json")[0]["count"]
print(f"The Insert completed! The index has {count} documents.")

Το ευρετήριο 'ir2026_index' δημιουργήθηκε.
Data Insert is starting. Please wait...
The Insert completed! The index has 40221 documents.
